# Task 2 · Contracts, nested data and a star schema

**Daniel Ehab** · 58-3825 · MET-02 · CSEN1095 Data Engineering, W26

The student clubs sell tickets to their events through an app, and the app sends every booking as one JSON document per line. This notebook keeps only the bookings that meet the contract (Part A), flattens them into tables (Part B), and models them as a star schema in DuckDB, saved as Parquet (Part C).

The input file is set once, in the first code cell, and every later cell reads it through `DATA`. Nothing here edits the file, and every count is computed from it, so the notebook runs unchanged on another file with the same shape.

In [1]:
DATA = "bookings.jsonl"

## Setup

The course's Python 3.12, with the packages pinned in `requirements.txt`: the same versions as Labs 4 and 5. The versions are printed so that anyone who re-runs the notebook can compare.

In [2]:
import datetime as dt
import json
import re
import sys
from collections import Counter
from typing import Literal

import duckdb
import ipykernel
import pandas as pd
import pyarrow
import pydantic
from pydantic import BaseModel, Field, ValidationError

print("Python    ", sys.version.split()[0])
for module in [ipykernel, pd, pydantic, duckdb, pyarrow]:
    print(f"{module.__name__:<10}", module.__version__)

Python     3.12.14
ipykernel  7.3.0
pandas     2.3.3
pydantic   2.13.5
duckdb     1.5.6
pyarrow    26.0.0


## Part A · The contract

### A.0 Look first

The brief shows one clean document. Before writing a single rule, we read every line of `DATA` and answer the first-look questions for nested data: what one line is, which keys it has, and which values each field takes.

The file is read once, as text, into `lines`. Nothing assigns to `lines` after this cell; every later step starts from it. Each line is parsed on its own, so a line that isn't valid JSON is counted instead of stopping the notebook, and every document keeps its line number.

In [3]:
with open(DATA, encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

docs = {}      # line number -> the parsed JSON value
not_json = {}  # line number -> the parser's error message
for n, line in enumerate(lines, start=1):
    try:
        docs[n] = json.loads(line)
    except json.JSONDecodeError as e:
        not_json[n] = str(e)

print("lines read:       ", len(lines))
print("parsed as JSON:   ", len(docs))
print("not valid JSON:   ", len(not_json), not_json)
print("not a JSON object:", sum(not isinstance(d, dict) for d in docs.values()))

lines read:        283
parsed as JSON:    283
not valid JSON:    0 {}
not a JSON object: 0


**One document**, the first one that parsed, printed with indentation so its nesting shows:

In [4]:
first = next(iter(docs.values()))
print(json.dumps(first, indent=2))

{
  "booking_id": "B0001",
  "booked_at": "2026-09-28T15:46:00Z",
  "student": {
    "id": "60-3778",
    "name": "Nour Mansour",
    "faculty": "IET"
  },
  "event": {
    "id": "EV-01",
    "title": "Robotics Demo Day",
    "club": "Robotics",
    "date": "2026-10-15",
    "venue": "Hall A"
  },
  "tickets": [
    {
      "type": "standard",
      "qty": 1,
      "price": 40.0
    },
    {
      "type": "vip",
      "qty": 3,
      "price": 100.0
    }
  ],
  "total": 340.0,
  "payment": "card",
  "status": "cancelled"
}


**Four small helpers** for looking inside the documents without trusting their shape:

- `at(doc, "student", "faculty")` walks a path and returns `"<no key>"` when a key is missing, so a missing key and a `null` value stay different;
- `tickets_of(doc)` returns the ticket list, or an empty list when `tickets` isn't a list;
- `shape(value)` describes a value: an object's sorted keys, or the type of anything else;
- `show(title, counter)` prints a `Counter`, most common first.

Values are counted by their `repr`, so text shows with its quotes and a hidden space can't hide (Task 1's `'Music '`).

In [5]:
def at(doc, *path):
    """The value at a path inside a document, or "<no key>" if a step is missing."""
    for key in path:
        if not isinstance(doc, dict) or key not in doc:
            return "<no key>"
        doc = doc[key]
    return doc


def tickets_of(doc):
    """The document's ticket list, or an empty list if it doesn't have one."""
    tickets = at(doc, "tickets")
    return tickets if isinstance(tickets, list) else []


def shape(value):
    """An object's sorted keys, or the type name of anything else ('NoneType' for null)."""
    return tuple(sorted(value)) if isinstance(value, dict) else type(value).__name__


def show(title, counter):
    print(title)
    for value, n in counter.most_common():
        print(f"  {n:>4}  {value}")


objects = [d for d in docs.values() if isinstance(d, dict)]
tickets = [t for d in objects for t in tickets_of(d)]
print(len(objects), "documents,", len(tickets), "ticket entries")

283 documents, 344 ticket entries


**Which keys does each level have?** Counted on the raw dicts, not on a DataFrame: once flattened, a missing key and a `null` look the same.

In [6]:
show("top level", Counter(shape(d) for d in objects))
show("student", Counter(shape(at(d, "student")) for d in objects))
show("event", Counter(shape(at(d, "event")) for d in objects))
show("tickets", Counter(shape(at(d, "tickets")) for d in objects))
show("each ticket", Counter(shape(t) for t in tickets))

top level
   193  ('booked_at', 'booking_id', 'event', 'payment', 'promo_code', 'status', 'student', 'tickets', 'total')
    90  ('booked_at', 'booking_id', 'event', 'payment', 'status', 'student', 'tickets', 'total')
student
   283  ('faculty', 'id', 'name')
event
   281  ('club', 'date', 'id', 'title', 'venue')
     2  NoneType
tickets
   283  list
each ticket
   344  ('price', 'qty', 'type')


**The fields with a fixed list of allowed values**, and how many tickets each booking holds:

In [7]:
show("student.faculty", Counter(repr(at(d, "student", "faculty")) for d in objects))
show("event.club", Counter(repr(at(d, "event", "club")) for d in objects))
show("ticket type", Counter(repr(at(t, "type")) for t in tickets))
show("payment", Counter(repr(at(d, "payment")) for d in objects))
show("status", Counter(repr(at(d, "status")) for d in objects))
show("tickets per booking", Counter(len(tickets_of(d)) for d in objects))

student.faculty
    75  'EMS'
    62  'MET'
    60  'IET'
    53  'Pharmacy'
    33  'Management'
event.club
    76  'Debate'
    74  'Music'
    56  'Robotics'
    40  'Chess'
    35  'Football'
     2  '<no key>'
ticket type
   187  'standard'
    78  'vip'
    76  'student'
     2  'early-bird'
     1  'premium'
payment
   145  'card'
    70  'cash'
    68  'wallet'
status
   257  'confirmed'
    26  'cancelled'
tickets per booking
   218  1
    63  2
     2  0


**The numbers**: their types, their range, and every value of `qty`.

In [8]:
def numbers(values):
    return [v for v in values if isinstance(v, (int, float)) and not isinstance(v, bool)]


for name, values in [("qty", [at(t, "qty") for t in tickets]),
                     ("price", [at(t, "price") for t in tickets]),
                     ("total", [at(d, "total") for d in objects])]:
    types = dict(Counter(type(v).__name__ for v in values))
    nums = numbers(values)
    print(f"{name:<6} types {types}  min {min(nums, default=None)}  max {max(nums, default=None)}")

show("qty values", Counter(repr(at(t, "qty")) for t in tickets))

qty    types {'int': 344}  min -1  max 4
price  types {'float': 344}  min 10.0  max 200.0
total  types {'float': 283}  min -30.0  max 850.0
qty values
   140  1
    99  2
    53  3
    49  4
     2  -1
     1  0


**Ids and dates**: each value with every digit replaced by `9`, so values of the same format collapse into one line.

In [9]:
def digits_as_9(value):
    return re.sub(r"\d", "9", value) if isinstance(value, str) else repr(value)


show("booking_id", Counter(digits_as_9(at(d, "booking_id")) for d in objects))
show("student.id", Counter(digits_as_9(at(d, "student", "id")) for d in objects))
show("booked_at", Counter(digits_as_9(at(d, "booked_at")) for d in objects))
show("event.date", Counter(digits_as_9(at(d, "event", "date")) for d in objects))

booking_id
   283  B9999
student.id
   279  99-9999
     2  99-99999
     1  999999
     1  99_9999
booked_at
   283  9999-99-99T99:99:99Z
event.date
   281  9999-99-99
     2  <no key>


**Does `booking_id` repeat?** The brief warns that the app sometimes sends a booking twice.

In [10]:
ids = Counter(repr(at(d, "booking_id")) for d in objects)
repeated = {i: n for i, n in ids.items() if n > 1}
print("documents:", len(objects), "| distinct booking_id:", len(ids), "| ids that repeat:", len(repeated))
print(repeated)

documents: 283 | distinct booking_id: 280 | ids that repeat: 3
{"'B0035'": 2, "'B0066'": 2, "'B0107'": 2}


**What we see** in `bookings.jsonl`, and where each finding is handled. The counts are in the outputs above; this cell names none, because a file with different problems would change them.

| The output shows | The brief's rule | Handled in |
|---|---|---|
| Every line parses as a JSON object | Validate every line | A.2 still quarantines a line that doesn't |
| Two document shapes: some carry `promo_code`, which the example lacks | Part B.3: find the drifting field | A.1 declares it, B.3 finds when it appears |
| Some documents have `"event": null` | `event` present, with all five fields | A.1, booking model |
| Some bookings have an empty ticket list | At least one ticket | A.1, booking model |
| Ticket types `'early-bird'` and `'premium'` | `type` one of `standard`, `vip`, `student` | A.1, ticket model |
| `qty` of `0` and `-1` | `qty` at least 1 | A.1, ticket model |
| `student.id` in three other formats (`99-99999`, `999999`, `99_9999`) | Two digits, a dash, four digits | A.1, student model |
| A negative `total` | `total` equals the sum of `qty × price` | A.1's model validator decides |
| Faculty, club, payment and status hold only allowed values, spelled exactly | One of the allowed values | A.1 writes the rules anyway, for the second file |
| `booked_at` has one format, ISO 8601 in UTC with a `Z`; `event.date` is always `YYYY-MM-DD` | `booked_at` before the event's date | A.1 types them as a datetime and a date |
| Some `booking_id`s appear twice | Keep the first copy | A.3 |
| Most bookings hold one ticket entry, some hold two | | B.1: `booking_tickets` has more rows than `bookings` |

Two rules can't be seen by counting values: whether each total adds up, and whether anyone booked after the event. Both compare fields inside one booking, so they wait for A.1's model validator.

### A.1 The contract as Pydantic models

The brief's table is the contract. Each rule becomes a type or a `Field` constraint, so it is written once, in a model, and Pydantic checks it on every document.

The rules come from the brief, not from what A.0 found in this file, because the graders run the notebook again on a file with different problems. What A.0 found becomes probes instead: broken values we feed to the models to watch each rule refuse them.

Pydantic runs in its default **lax mode** (Lab 4): it converts a value when nothing is lost (`"2"` becomes `2` for an `int`) and refuses when something would be (`2.5`, `"two"`).

First the three small models. The booking model that uses them, with the two rules that compare fields, comes after them.

| The brief's rule | Where it lives |
|---|---|
| `student.id`: two digits, a dash, four digits | `Student.id`, `pattern=r"^\d{2}-\d{4}$"` |
| `student.faculty`: one of `MET`, `IET`, `EMS`, `Pharmacy`, `Management` | `Student.faculty`, the `Faculty` literal |
| `event`: all five fields; `club` one of `Robotics`, `Debate`, `Music`, `Football`, `Chess` | `Event`: five required fields, `club` the `Club` literal |
| Each ticket: `type` one of `standard`, `vip`, `student`; `qty` at least 1; `price` not negative | `Ticket`: the `TicketType` literal, `ge=1`, `ge=0` |
| `event` present, at least one ticket, `total`, `booked_at`, `payment`, `status` | The booking model, next |

In [11]:
Faculty = Literal["MET", "IET", "EMS", "Pharmacy", "Management"]
Club = Literal["Robotics", "Debate", "Music", "Football", "Chess"]
TicketType = Literal["standard", "vip", "student"]


class Ticket(BaseModel):
    type: TicketType
    qty: int = Field(ge=1)      # at least 1
    price: float = Field(ge=0)  # not negative, so a free ticket is allowed


class Student(BaseModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")  # two digits, a dash, four digits, and nothing else
    name: str
    faculty: Faculty


class Event(BaseModel):
    id: str
    title: str
    club: Club
    date: dt.date  # "2026-10-15" becomes a real date, so the booking model can compare it
    venue: str

**Probes**, like Lab 4's check cells. The good examples are the brief's own example document; each probe then breaks one rule, and Pydantic must refuse it and name that field. The broken values include the ones A.0 found in this file (`'early-bird'`, `'premium'`, a `qty` of `0` and `-1`, the three wrong id formats) and others it didn't.

`refused(model, data)` returns the fields Pydantic names when it refuses the data, or `None` when it accepts it.

In [12]:
def refused(model, data):
    """The fields Pydantic names when it refuses data, or None when it accepts it."""
    try:
        model.model_validate(data)
        return None
    except ValidationError as e:
        return sorted({".".join(map(str, err["loc"])) for err in e.errors()})


good_ticket = {"type": "standard", "qty": 1, "price": 40.0}
good_student = {"id": "61-3826", "name": "Laila Adel", "faculty": "IET"}
good_event = {"id": "EV-01", "title": "Robotics Demo Day", "club": "Robotics", "date": "2026-10-15", "venue": "Hall A"}

PROBES = [  # (what the probe does, model, data, the field Pydantic must name, or None if it must pass)
    ("the brief's ticket", Ticket, good_ticket, None),
    ("type 'early-bird'", Ticket, {**good_ticket, "type": "early-bird"}, "type"),
    ("type 'premium'", Ticket, {**good_ticket, "type": "premium"}, "type"),
    ("type 'VIP', in capitals", Ticket, {**good_ticket, "type": "VIP"}, "type"),
    ("qty 0", Ticket, {**good_ticket, "qty": 0}, "qty"),
    ("qty -1", Ticket, {**good_ticket, "qty": -1}, "qty"),
    ("qty 2.5", Ticket, {**good_ticket, "qty": 2.5}, "qty"),
    ("qty '2', as text", Ticket, {**good_ticket, "qty": "2"}, None),
    ("price 0, a free ticket", Ticket, {**good_ticket, "price": 0}, None),
    ("price -5", Ticket, {**good_ticket, "price": -5.0}, "price"),
    ("ticket without price", Ticket, {"type": "vip", "qty": 1}, "price"),
    ("the brief's student", Student, good_student, None),
    ("id '61-38261', a fifth digit", Student, {**good_student, "id": "61-38261"}, "id"),
    ("id '613826', no dash", Student, {**good_student, "id": "613826"}, "id"),
    ("id '61_3826', an underscore", Student, {**good_student, "id": "61_3826"}, "id"),
    ("id 'ID61-3826', text in front", Student, {**good_student, "id": "ID61-3826"}, "id"),
    ("faculty 'iet', lower case", Student, {**good_student, "faculty": "iet"}, "faculty"),
    ("the brief's event", Event, good_event, None),
    ("event without venue", Event, {k: v for k, v in good_event.items() if k != "venue"}, "venue"),
    ("venue null", Event, {**good_event, "venue": None}, "venue"),
    ("club 'Chess Club'", Event, {**good_event, "club": "Chess Club"}, "club"),
    ("date '15/10/2026'", Event, {**good_event, "date": "15/10/2026"}, "date"),
]

probe_results = pd.DataFrame([
    {"probe": label, "model": model.__name__, "must name": must or "(passes)",
     "Pydantic names": ", ".join(refused(model, data) or ["(passes)"])}
    for label, model, data, must in PROBES
])
probe_results["ok"] = probe_results["must name"] == probe_results["Pydantic names"]
probe_results

,probe,model,must name,Pydantic names,ok
0,the brief's ticket,Ticket,(passes),(passes),True
1,type 'early-bird',Ticket,type,type,True
2,type 'premium',Ticket,type,type,True
3,"type 'VIP', in capitals",Ticket,type,type,True
4,qty 0,Ticket,qty,qty,True
5,qty -1,Ticket,qty,qty,True
6,qty 2.5,Ticket,qty,qty,True
7,"qty '2', as text",Ticket,(passes),(passes),True
8,"price 0, a free ticket",Ticket,(passes),(passes),True
9,price -5,Ticket,price,price,True


In [13]:
# ✓ check
assert probe_results["ok"].all(), probe_results[~probe_results["ok"]]
print("✓", len(PROBES), "probes: each good example passes, and each broken rule is refused for its own field")

✓ 22 probes: each good example passes, and each broken rule is refused for its own field


**Why `^` and `$` in the id pattern?** The same pattern without them, on two ids that break the rule:

In [14]:
class StudentWithoutAnchors(BaseModel):
    id: str = Field(pattern=r"\d{2}-\d{4}")


for student_id in ["61-38261", "ID61-3826"]:
    print(f"{student_id!r:12} without anchors: {refused(StudentWithoutAnchors, {'id': student_id}) or 'passes':10}"
          f"  with anchors: {refused(Student, {**good_student, 'id': student_id}) or 'passes'}")

'61-38261'   without anchors: passes      with anchors: ['id']
'ID61-3826'  without anchors: passes      with anchors: ['id']


Without anchors both pass: Pydantic's `pattern` looks for a match **anywhere** in the text, like `re.search`, so `61-3826` hidden inside a longer value is enough. `^` (start) and `$` (end) make the whole value match the rule.